In [1]:
import sys
from pathlib import Path

parent_folder = Path.cwd().parent
if str(parent_folder) not in sys.path:
    sys.path.append(str(parent_folder))
    
import warnings
warnings.filterwarnings('ignore')

import numpy as np
from simulation import Simulation
import xtrack as xt

In [2]:
sim = Simulation(threads=10,folder=Path('../Lattice_Files/02_Aperture_Lattice_mod/'))
line = sim.line

Converting sequence "synchrotron":   0%|          | 0/1990 [00:00<?, ?it/s]

In [3]:
line.vars['kqf'] = 0#-0.024
line.vars['kqd'] = 0#0.065
for name in line.element_names:
    element = line[name]
    
    if isinstance(element, xt.Quadrupole):
        if 'qtf' in name.lower():
            line.element_refs[name].k1 += line.vars['kqf']
            print(f"Linked {name} to kqf")
            
        elif 'qtd' in name.lower():
            line.element_refs[name].k1 += line.vars['kqd']
            print(f"Linked {name} to kqd")

print('Done!')

Done!


In [4]:
opt = line.match(
    method='4d', 
    freeze_longitudinal=True,
    vary=[xt.VaryList(['kqd'], step=1e-9), 
          xt.VaryList(['kqf'], step=1e-9)],
    targets = [xt.TargetSet(qx=4.47, qy=3.74, tol=1e-6, tag='tune')]
)

print(opt.get_knob_values())

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.
                                             
Optimize - start penalty: 1.56                              
Matching: model call n. 16 penalty = 7.3194e-06              
Optimize - end penalty:  7.31938e-06                            
{'kqd': np.float64(-0.01591752996940869), 'kqf': np.float64(0.04882362621710519)}


In [30]:
k_vars = [v for v in line.vars.keys() if v.startswith('k')]
print(k_vars)

['kqf', 'kqd']


In [37]:
qf_count = sum(1 for n in line.element_names if 'qf' in n.lower() and isinstance(line[n], xt.Quadrupole))
qd_count = sum(1 for n in line.element_names if 'qd' in n.lower() and isinstance(line[n], xt.Quadrupole))
print(f"Linked {qf_count} QF magnets and {qd_count} QD magnets.")

Linked 140 QF magnets and 250 QD magnets.
